In [1]:
import os
# import jupytergis
# _tiler_src = "/home/greg/Documents/programming/quantstack/jupytergis-tiler/src/jupytergis"
# if _tiler_src not in jupytergis.__path__:
# 	jupytergis.__path__.append(_tiler_src)
# os.environ["AWS_VIRTUAL_HOSTING"] = "FALSE"
# os.environ["AWS_ENDPOINT_URL"] = "https://eodata.dataspace.copernicus.eu"
os.environ["AWS_ACCESS_KEY_ID"] = "HGWDKN4N26XVYNYMRXP0"
os.environ["AWS_SECRET_ACCESS_KEY"] = "R893NB8q4vTK1NaZ7BH6Aa5Yq043O6zJ9lHE96sn"
# os.environ["AWS_DEFAULT_REGION"] = "default"   
# os.environ["AWS_REGION"] = "eu-central-1"
# if that still gives signature errors, try:
# os.environ["AWS_REGION"] = "us-east-1"


os.environ["GDAL_HTTP_TCP_KEEPALIVE"] = "YES"
os.environ["AWS_S3_ENDPOINT"] = "eodata.dataspace.copernicus.eu"
# os.environ["AWS_ACCESS_KEY_ID"] = os.environ.get("CDSE_S3_ACCESS_KEY")
# os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ.get("CDSE_S3_SECRET_KEY")
os.environ["AWS_HTTPS"] = "YES"
os.environ["AWS_VIRTUAL_HOSTING"] = "FALSE"
os.environ["GDAL_HTTP_UNSAFESSL"] = "YES"




In [2]:
import stackstac

from jupytergis.tiler import GISDocument

doc = GISDocument()

doc.add_raster_layer(
    url="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    name="Google Satellite",
    attribution="Google",
    opacity=0.6,
)

doc

In [18]:
item = doc.stacItem

In [19]:
da = stackstac.stack(item, epsg=3857)

In [20]:
green = da.sel(band='B03_10m')
nir = da.sel(band='B08_10m')
ndwi = (green - nir) / (green + nir)
ndwi = ndwi.where((green + nir) != 0)
ndwi.name = 'ndwi'

In [21]:
import dask.diagnostics
with dask.diagnostics.ProgressBar():
    data = ndwi.compute()

[########################################] | 100% Completed | 72.21 s


In [22]:
vmin_acc, vmax_acc = int(data.min().compute()), int(data.max().compute())
vmin_acc, vmax_acc

(0, 1)

In [23]:
data.ndim

3

In [24]:
out = data.isel(time=0)

In [26]:
await doc.add_tiler_layer(
    name="ndwi",
    data_array=out,
    colormap_name="plasma",
    rescale=(vmin_acc, vmax_acc),
    reproject="max",
)

'26a685dc-876c-4b5d-9610-fac7128e8676'